# 🐧 Palmer Penguins — R Practice Notebook
### HPC Summer Workshop 2026

> **Dataset:** Palmer Penguins — morphological measurements of 344 penguins  
> collected at three islands in the Palmer Archipelago, Antarctica (2007–2009).  
> Three species: *Adelie*, *Chinstrap*, *Gentoo*.  
> **Source:** Gorman, Williams & Fraser (2014) • [allisonhorst.github.io/palmerpenguins](https://allisonhorst.github.io/palmerpenguins/)

**What you will practice:**
| Section | Skills |
|---------|--------|
| 1 · Setup & Loading | `readr`, package installation, first look |
| 2 · Exploration | `glimpse`, `summary`, `table`, missing data |
| 3 · Data Wrangling | `dplyr` verbs: `filter`, `select`, `mutate`, `group_by`, `summarise` |
| 4 · Visualization | `ggplot2` scatter, box, bar, facet plots |
| 5 · Statistics | t-tests, correlation, linear models |
| 6 · Machine Learning | k-NN classification, train/test split |
| 7 · Parallelism | `parallel` / `furrr` for bootstrap resampling |
| 8 · AI Collaboration | Prompt engineering with R tasks |

---
⚠️ **Each exercise cell is marked** `# YOUR CODE HERE`.  
💡 **Hints** are available — click the ▶ triangle to expand them.  
🔒 A separate **solutions notebook** is provided for self-checking.


## Section 1 — Setup & Data Loading

### 1.1 Install & load packages

We need: `tidyverse` (dplyr + ggplot2 + readr), `caret` (ML), `furrr` (parallel map).  
Run the install cell once; it is safe to re-run.


In [ ]:
# Install packages (run once — safe to re-run)
pkgs <- c("tidyverse", "caret", "furrr", "future", "broom", "patchwork")
new_pkgs <- pkgs[!pkgs %in% installed.packages()[,"Package"]]
if (length(new_pkgs)) install.packages(new_pkgs, repos = "https://cloud.r-project.org")

# Load core libraries
library(tidyverse)
library(broom)
library(patchwork)
cat("✅ Packages loaded\n")


### 1.2 Load the data

**Exercise 1 ·** Load the penguins CSV from the URL below into a tibble called `penguins`.  
Print the first 6 rows with `head()`.

```
URL: https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv
```


<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

Use `read_csv("URL")` from the `readr` package (loaded with tidyverse).  
Assign with `<-` and inspect with `head(penguins)`.


</details>

In [ ]:
# YOUR CODE HERE


### 1.3 Quick look

**Exercise 2 ·** Use `glimpse()` to see column types, then `summary()` for statistics.  
How many rows and columns does the dataset have?  
What are the three penguin species present?


<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

`glimpse(penguins)` and `summary(penguins)` — also try `nrow()`, `ncol()`, `names()`.  
For a quick species count: `table(penguins$species)`.


</details>

In [ ]:
# YOUR CODE HERE


## Section 2 — Exploratory Data Analysis

### 2.1 Missing values

**Exercise 3 ·** Count the number of `NA` values in each column.  
Which columns have missing data, and how many rows have *any* missing value?


<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

```r
colSums(is.na(penguins))          # NAs per column
sum(!complete.cases(penguins))    # rows with any NA
```


</details>

In [ ]:
# YOUR CODE HERE


### 2.2 Distribution by species & island

**Exercise 4 ·** Build a two-way frequency table of `species` vs `island`.  
Which species is found on all three islands? Which are restricted to one?


<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

`table(penguins$species, penguins$island)` — or use `count()` from dplyr:  
`penguins |> count(species, island)`.


</details>

In [ ]:
# YOUR CODE HERE


### 2.3 Body mass by sex

**Exercise 5 ·** Filter out rows where `sex` is `NA`.  
Then compute mean `body_mass_g` grouped by `species` **and** `sex`.  
Arrange the result by mean body mass descending.


<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

```r
penguins |>
  filter(!is.na(sex)) |>
  group_by(species, sex) |>
  summarise(mean_mass = mean(body_mass_g, na.rm = TRUE)) |>
  arrange(desc(mean_mass))
```


</details>

In [ ]:
# YOUR CODE HERE


## Section 3 — Data Wrangling with dplyr

### 3.1 Creating new columns

**Exercise 6 ·** Add two new columns to `penguins`:
- `bill_ratio` = `bill_length_mm / bill_depth_mm`
- `size_class` = `"large"` if `body_mass_g >= 4500`, else `"small"`

Store the result as `penguins_aug`.  
Show how many penguins fall into each size class per species.


<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

Use `mutate()` for both columns. For `size_class`, use `ifelse()` or `case_when()`:
```r
mutate(size_class = ifelse(body_mass_g >= 4500, "large", "small"))
```
Then `count(penguins_aug, species, size_class)`.


</details>

In [ ]:
# YOUR CODE HERE


### 3.2 Pivoting & reshaping

**Exercise 7 ·** Select only `species`, `bill_length_mm`, `bill_depth_mm`,  
`flipper_length_mm`, and `body_mass_g` from `penguins`.  
Pivot to *long* format so you have columns `species`, `measurement`, `value`.  
Then compute mean and standard deviation of each measurement per species.


<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

```r
penguins |>
  select(species, bill_length_mm:body_mass_g) |>
  pivot_longer(-species, names_to = "measurement", values_to = "value") |>
  group_by(species, measurement) |>
  summarise(mean = mean(value, na.rm=TRUE), sd = sd(value, na.rm=TRUE))
```


</details>

In [ ]:
# YOUR CODE HERE


### 3.3 Joining datasets

**Exercise 8 ·** Create a small lookup tibble:

```r
island_info <- tibble(
  island   = c("Torgersen", "Biscoe", "Dream"),
  region   = c("Antarctic Peninsula", "Biscoe Islands", "Dream Island"),
  approx_area_km2 = c(5.4, 64.0, 3.7)
)
```

Left-join this into `penguins` and display one row per island with its area  
alongside the average `body_mass_g` for penguins observed there.


<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

```r
penguins |>
  left_join(island_info, by = "island") |>
  group_by(island, region, approx_area_km2) |>
  summarise(avg_mass = mean(body_mass_g, na.rm=TRUE))
```


</details>

In [ ]:
island_info <- tibble(
  island          = c("Torgersen", "Biscoe", "Dream"),
  region          = c("Antarctic Peninsula", "Biscoe Islands", "Dream Island"),
  approx_area_km2 = c(5.4, 64.0, 3.7)
)

# YOUR CODE HERE


## Section 4 — Visualisation with ggplot2

### 4.1 Scatter plot — bill dimensions

**Exercise 9 ·** Create a scatter plot of `bill_length_mm` (x) vs `bill_depth_mm` (y).  
Color points by `species` and add a `geom_smooth(method = "lm")` per species.  
Remove NA rows first. Add clear axis labels and a title.

> **Question:** Do the three species form distinct clusters?  
> What does the negative overall slope (ignoring species) illustrate  
> about Simpson's Paradox?


<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

```r
penguins |>
  filter(complete.cases(bill_length_mm, bill_depth_mm, species)) |>
  ggplot(aes(bill_length_mm, bill_depth_mm, color = species)) +
  geom_point(alpha = 0.7) +
  geom_smooth(method = "lm", se = FALSE) +
  labs(title = "...", x = "...", y = "...")
```


</details>

In [ ]:
# YOUR CODE HERE


### 4.2 Box plot — body mass by species & sex

**Exercise 10 ·** Make a box plot of `body_mass_g` by `species`, filled by `sex`  
(exclude NA sex rows). Use `theme_minimal()` and `scale_fill_manual()` with  
two colors of your choice.


<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

```r
penguins |> filter(!is.na(sex)) |>
  ggplot(aes(species, body_mass_g, fill = sex)) +
  geom_boxplot() +
  scale_fill_manual(values = c("darkorange","steelblue")) +
  theme_minimal()
```


</details>

In [ ]:
# YOUR CODE HERE


### 4.3 Faceted histograms

**Exercise 11 ·** Plot histograms of `flipper_length_mm` faceted by `species`  
(one panel per species). Use `binwidth = 5`. Color the fill by island.  
Use `facet_wrap(~ species, ncol = 1)`.


<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

```r
penguins |> filter(!is.na(flipper_length_mm)) |>
  ggplot(aes(flipper_length_mm, fill = island)) +
  geom_histogram(binwidth = 5, color = "white") +
  facet_wrap(~ species, ncol = 1) +
  theme_minimal()
```


</details>

In [ ]:
# YOUR CODE HERE


### 4.4 Patchwork — combine plots

**Exercise 12 ·** Using the `patchwork` package, combine the scatter plot  
from 4.1 and the box plot from 4.2 side by side.  
Add a shared title with `plot_annotation(title = "Palmer Penguins Summary")`.


<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

Store each plot as `p1` and `p2`, then:
```r
library(patchwork)
p1 + p2 + plot_annotation(title = "Palmer Penguins Summary")
```


</details>

In [ ]:
# YOUR CODE HERE


## Section 5 — Statistical Analysis

### 5.1 Two-sample t-test

**Exercise 13 ·** Test whether mean `body_mass_g` differs significantly  
between **male** and **female** Gentoo penguins.  
State H₀ and H₁, run a Welch t-test, and interpret the p-value and 95% CI.


<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

```r
gentoo <- penguins |> filter(species == "Gentoo", !is.na(sex))
t.test(body_mass_g ~ sex, data = gentoo)
```
H₀: μ_male = μ_female.  H₁: μ_male ≠ μ_female.


</details>

In [ ]:
# YOUR CODE HERE


### 5.2 Correlation matrix

**Exercise 14 ·** Compute a Pearson correlation matrix for the four numeric  
measurement columns (`bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`,  
`body_mass_g`). Which pair is most strongly correlated?  
Visualise it with a simple heatmap using `ggplot2`.


<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

```r
nums <- penguins |> select(bill_length_mm:body_mass_g) |> drop_na()
cor_mat <- cor(nums)
# reshape for ggplot:
cor_mat |> as.data.frame() |> rownames_to_column("var1") |>
  pivot_longer(-var1, names_to="var2", values_to="r") |>
  ggplot(aes(var1, var2, fill=r)) + geom_tile() +
  scale_fill_gradient2(low="blue", high="red", mid="white", midpoint=0) +
  geom_text(aes(label=round(r,2))) + theme_minimal()
```


</details>

In [ ]:
# YOUR CODE HERE


### 5.3 Linear regression

**Exercise 15 ·** Fit a linear model predicting `body_mass_g` from  
`flipper_length_mm`, `species`, and `sex` (drop NAs first).  
Use `tidy()` from `broom` to display the coefficients.  
Which predictor has the largest effect?  What is the R² value?


<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

```r
model <- lm(body_mass_g ~ flipper_length_mm + species + sex,
            data = drop_na(penguins, body_mass_g, flipper_length_mm, species, sex))
tidy(model)
glance(model)  # R² and other model-level stats
```


</details>

In [ ]:
# YOUR CODE HERE


## Section 6 — Machine Learning: Classify Penguin Species

### 6.1 Prepare data

**Exercise 16 ·** Prepare a clean ML-ready tibble `penguins_ml` with:
- Only complete cases (no NAs in any column)
- Columns: `species`, `bill_length_mm`, `bill_depth_mm`,  
  `flipper_length_mm`, `body_mass_g`
- `species` converted to a factor

Split into 80 % training and 20 % test sets using `set.seed(42)`.  
Report the class balance in the training set.


<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

```r
set.seed(42)
penguins_ml <- penguins |>
  select(species, bill_length_mm:body_mass_g) |>
  drop_na() |>
  mutate(species = factor(species))

idx   <- sample(nrow(penguins_ml), 0.8 * nrow(penguins_ml))
train <- penguins_ml[idx, ]
test  <- penguins_ml[-idx, ]
table(train$species)
```


</details>

In [ ]:
# YOUR CODE HERE


### 6.2 Train a k-NN model

**Exercise 17 ·** Using `caret`, train a k-Nearest Neighbour classifier  
(`method = "knn"`) to predict `species` from the four numeric features.  
Use 5-fold cross-validation (`trainControl`).  
Tune `k` over `c(3, 5, 7, 9, 11)`.


<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

```r
library(caret)
ctrl  <- trainControl(method = "cv", number = 5)
kgrid <- expand.grid(k = c(3, 5, 7, 9, 11))
knn_model <- train(species ~ ., data = train,
                   method    = "knn",
                   trControl = ctrl,
                   tuneGrid  = kgrid,
                   preProcess = c("center", "scale"))
print(knn_model)
```


</details>

In [ ]:
# YOUR CODE HERE


### 6.3 Evaluate on test set

**Exercise 18 ·** Predict species for the test set and print:
- A confusion matrix (`confusionMatrix()`)
- Overall accuracy, and per-class Sensitivity / Specificity

Which species is hardest to classify correctly?


<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

```r
preds <- predict(knn_model, newdata = test)
confusionMatrix(preds, test$species)
```


</details>

In [ ]:
# YOUR CODE HERE


## Section 7 — Parallel Computing: Bootstrap Confidence Intervals

**Exercise 19 ·** Estimate a 95 % bootstrap CI for the mean `flipper_length_mm`  
of Gentoo penguins using **1000 bootstrap resamples**.

1. Do it *sequentially* first and record wall-clock time (`system.time`).
2. Repeat using **`furrr::future_map_dbl()`** with  
   `plan(multisession, workers = 4)` for parallel execution.
3. Compare timings and report the CI from the parallel run.

> **HPC note:** On a cluster, `workers` can match your SLURM `--cpus-per-task`.


<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

```r
library(furrr)
library(future)

gentoo_flip <- penguins |>
  filter(species == "Gentoo", !is.na(flipper_length_mm)) |>
  pull(flipper_length_mm)

set.seed(99)
boot_fn <- function(i) mean(sample(gentoo_flip, replace = TRUE))

# Sequential
t_seq <- system.time(
  boot_seq <- sapply(1:1000, boot_fn)
)

# Parallel
plan(multisession, workers = 4)
t_par <- system.time(
  boot_par <- future_map_dbl(1:1000, boot_fn)
)
plan(sequential)  # reset

# CI
quantile(boot_par, c(0.025, 0.975))
```


</details>

In [ ]:
# YOUR CODE HERE


## Section 8 — AI Collaboration in Data Science

### 8.1 Prompt engineering for R code

**Exercise 20 ·** Below are **three poorly written prompts** asking an AI assistant  
to help with this dataset. *Rewrite each one* to be specific, context-rich,  
and likely to produce correct, runnable R code.

| # | Poor prompt | Your improved prompt |
|---|-------------|----------------------|
| A | *"Clean the data"* | *(write here)* |
| B | *"Make a nice chart"* | *(write here)* |
| C | *"Do statistics on penguins"* | *(write here)* |

> **Discussion:** What elements make a data-science prompt effective?  
> (Dataset context? Output format? Preferred packages? Error messages?)


In [ ]:
# This is a reflection / markdown exercise.
# Write your improved prompts as comments or in the markdown cell above.

# Improved Prompt A:
# ...

# Improved Prompt B:
# ...

# Improved Prompt C:
# ...


### 8.2 Debug with AI assistance

**Exercise 21 ·** The code cell below contains **four bugs**.  
Try to run it, read the error messages, and fix all bugs.  
Then describe how you would use an AI assistant efficiently to debug code —  
what information would you include in your prompt?


<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

Bugs: (1) wrong function name, (2) incorrect column name,  
(3) missing `na.rm = TRUE`, (4) wrong grouping variable name.  
When prompting an AI: always paste the **full error message** + the **failing code block**,  
and state the expected vs. actual output.


</details>

In [ ]:
# BUGGY CODE — find and fix all 4 bugs
library(Tidyverse)   # Bug 1

penguins |>
  filter(!is.na(Sex)) |>           # Bug 2
  group_by(species, sex) |>
  summarise(
    avg_flipper = average(flipper_length_mm),   # Bug 3
    n = n()
  ) |>
  arrange(desc(avg_flippr))        # Bug 4


### 8.3 AI-generated code review

**Exercise 22 ·** An AI assistant produced the following code in response to  
*"Plot bill length vs body mass for each species"*.  
Review it critically: is it correct? Could it be improved?  
Rewrite it with at least two improvements.

```r
ggplot(penguins) +
  geom_point(aes(x = bill_length, y = body_mass)) +
  facet_wrap(species)
```


<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

Issues to spot: (1) wrong column names (should be `bill_length_mm`, `body_mass_g`),  
(2) `facet_wrap` needs `~ species`, (3) no axis labels, (4) no NA handling,  
(5) could add color for clarity.


</details>

In [ ]:
# YOUR IMPROVED CODE HERE


---
## 🎉 Well done!

You have worked through data loading, wrangling, visualisation, statistics,  
machine learning, parallel computing, and AI-assisted coding — all with the  
Palmer Penguins dataset.

### Challenge Extensions (optional)
- 🔬 Add `penguins_raw.csv` and explore the isotope columns (`delta_15N`, `delta_13C`)
- 📊 Build an interactive plot with `plotly::ggplotly()`
- 🌲 Replace k-NN with a Random Forest; compare accuracy
- ⚡ Scale the bootstrap to 10,000 resamples and profile on your HPC cluster

**Solutions notebook:** `Palmer_Penguins_Solutions.ipynb`
